# Week 7, Day 5: How many dimensions? Spectra, noise and cross-validation

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2019)
N, S = 100, 600                              # neurons, stimuli
n = np.arange(1, N + 1)
true_spec = n ** -1.0                         # a power law with exponent 1
Q = np.linalg.qr(rng.normal(size=(N, N)))[0]  # random orthogonal axes in neural space
signal = (rng.normal(size=(S, N)) * np.sqrt(true_spec)) @ Q.T
R1 = signal + 0.3 * rng.normal(size=(S, N))   # first presentation of every stimulus
R2 = signal + 0.3 * rng.normal(size=(S, N))   # second presentation: same signal, new noise
print("total signal variance", true_spec.sum().round(2), "  total noise variance", round(N * 0.3 ** 2, 2))

### Exercise: Ordinary PCA of one repeat

Compute the eigenvalues of the covariance of `R1`, largest first, in `lam_naive`. Define `participation_ratio` (yesterday's formula) and compute it for the true spectrum (`pr_true`) and for the naive one (`pr_naive`). Plot both spectra on log-log axes.

In [ ]:
def participation_ratio(lam):
    return ...

lam_naive = ...
pr_true, pr_naive = ..., ...

plt.figure(figsize=(5, 3.2))
plt.loglog(n, true_spec, label="true signal"); plt.loglog(n, lam_naive, label="ordinary PCA, one repeat")
plt.axhline(0.3 ** 2, ls="--", color="0.5", label="noise variance"); plt.xlabel("component"); plt.ylabel("variance"); plt.legend(frameon=False); plt.show()
print("participation ratio: true", round(pr_true, 1), " ordinary PCA", round(pr_naive, 1))

In [ ]:
import numpy as np
def test_naive():
    "lam_naive holds the covariance eigenvalues of R1, largest first"
    assert np.allclose(lam_naive, np.sort(np.linalg.eigvalsh(np.cov(R1, rowvar=False)))[::-1])

def test_pr():
    "participation_ratio is defined, and the naive PR is far above the true one"
    assert np.isclose(participation_ratio(np.ones(4)), 4)
    assert pr_naive > 2 * pr_true

test_naive()
test_pr()
print("All checks passed")

### Exercise: Implement cross-validated PCA

Write `cvpca(R1, R2, train, test)`: take the PCs (rows of `Vt`) of the centred first presentation of the training stimuli, then project the centred first and second presentations of the test stimuli onto them and return the mean product of the two projections for each component. Compute `cv_spec` with a random half split and its participation ratio `pr_cv` (clip negative values to zero first, since noise can make them slightly negative).

In [ ]:
def cvpca(R1, R2, train, test):
    A = R1[train] - R1[train].mean(axis=0)
    Vt = np.linalg.svd(A, full_matrices=False)[2]
    B1 = ...
    B2 = ...
    return ...

order = rng.permutation(S)
train, test = order[: S // 2], order[S // 2 :]
cv_spec = cvpca(R1, R2, train, test)
pr_cv = ...

plt.figure(figsize=(5, 3.2))
plt.loglog(n, true_spec, label="true signal"); plt.loglog(n, lam_naive, label="ordinary PCA")
plt.loglog(n, np.clip(cv_spec, 1e-4, None), "o", ms=3, label="cross-validated PCA")
plt.xlabel("component"); plt.ylabel("variance"); plt.legend(frameon=False); plt.show()
print("participation ratio: true", round(pr_true, 1), " ordinary", round(pr_naive, 1), " cross-validated", round(pr_cv, 1))

In [ ]:
import numpy as np
def test_shape():
    "cvpca returns one value per neuron"
    assert cv_spec.shape == (N,)

def test_close():
    "the cumulative cross-validated variance of the first 20 components is within 20% of the truth"
    ratio = np.cumsum(cv_spec)[:20] / np.cumsum(true_spec)[:20]
    assert np.all(np.abs(ratio - 1) < 0.2), ratio.round(2)

def test_no_floor():
    "the cross-validated tail has no noise floor: components 40 to 100 average below 0.04"
    assert np.mean(cv_spec[40:]) < 0.04 and np.mean(lam_naive[40:]) > 0.08

def test_pr():
    "the cross-validated participation ratio is within 25% of the true one"
    assert abs(pr_cv - pr_true) / pr_true < 0.25

test_shape()
test_close()
test_no_floor()
test_pr()
print("All checks passed")

### Exercise: Fit the exponent

Write `fit_alpha(spec, lo, hi)` that fits a straight line to log variance against log component number for components `lo` to `hi` (1-based, inclusive of `lo`, exclusive of `hi`), using only positive values, and returns minus the slope. Fit components 2 to 40 of the cross-validated and naive spectra.

In [ ]:
def fit_alpha(spec, lo, hi):
    k = np.arange(lo, hi)              # component numbers, 1-based
    y = spec[k - 1]
    keep = y > 0
    slope = ...
    return -slope

alpha_cv = fit_alpha(cv_spec, 2, 40)
alpha_naive = fit_alpha(lam_naive, 2, 40)
print("exponent: cross-validated", round(alpha_cv, 2), " naive", round(alpha_naive, 2), " true 1.0")

In [ ]:
import numpy as np
def test_fit():
    "fit_alpha recovers the exponent of an exact power law"
    k = np.arange(1, 101)
    assert np.isclose(fit_alpha(k ** -1.5, 2, 40), 1.5)

def test_estimates():
    "the cross-validated exponent is near 1 and the naive one far below"
    assert abs(alpha_cv - 1) < 0.25, alpha_cv
    assert alpha_naive < 0.7

test_fit()
test_estimates()
print("All checks passed")